# Proyecto 3 — Optimización de Pricing en Rutas de Ferry

## Notebook 7 — Dashboard ejecutivo final

En este notebook construimos la **capa final de comunicación ejecutiva** del proyecto.

El objetivo es integrar todo lo construido en los notebooks anteriores:

- Dataset base.
- Análisis exploratorio.
- Elasticidad.
- Modelo predictivo de demanda.
- Simulación avanzada de escenarios.
- Recomendador formal de pricing.
- Herramientas interactivas.
- Dashboard HTML final.

Este notebook convierte el proyecto en un entregable completo de portfolio.

## 1. Objetivo del notebook

El dashboard debe responder de forma rápida:

- ¿Cuál es la situación global del negocio?
- ¿Qué rutas tienen mayor revenue, margen y ocupación?
- ¿Qué impacto estimado tienen las recomendaciones?
- ¿Qué acciones de pricing aparecen con más frecuencia?
- ¿Qué rutas requieren subida, promoción, mantenimiento o revisión?
- ¿Qué herramientas interactivas acompañan al análisis?

La lógica final es:

```text
Datos → análisis → modelo → simulación → recomendación → dashboard ejecutivo
```

In [ ]:
# ============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings

from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid")

print("Librerías importadas correctamente.")

In [ ]:
# ============================================================
# 2. CONEXIÓN CON GOOGLE DRIVE
# ============================================================

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive conectado correctamente.")
except:
    print("No estás ejecutando este notebook en Google Colab o Drive ya está montado.")

In [ ]:
# ============================================================
# 3. DEFINICIÓN DE RUTAS DEL PROYECTO
# ============================================================

base_path = "/content/drive/MyDrive/7 Colab Notebooks/1 Porfolio/Balearia/3 Pricing"

data_path = f"{base_path}/data/processed/ferry_pricing_dataset.csv"
reports_path = f"{base_path}/reports"
images_path = f"{base_path}/images"
models_path = f"{base_path}/models"
dashboard_path = f"{base_path}/dashboard"

os.makedirs(reports_path, exist_ok=True)
os.makedirs(images_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)
os.makedirs(dashboard_path, exist_ok=True)

print("Ruta base del proyecto:")
print(base_path)

## 2. Funciones auxiliares

Creamos funciones para:

- Cargar archivos de forma segura.
- Asegurar columnas numéricas.
- Exportar tablas.
- Formatear valores para el dashboard.

In [ ]:
# ============================================================
# 4. FUNCIONES AUXILIARES
# ============================================================

def force_numeric_columns(dataframe, columns):
    df_copy = dataframe.copy()
    for col in columns:
        if col not in df_copy.columns:
            continue
        direct = pd.to_numeric(df_copy[col], errors="coerce")
        if direct.notna().mean() >= 0.80:
            df_copy[col] = direct
        else:
            df_copy[col] = (
                df_copy[col]
                .astype(str)
                .str.replace("€", "", regex=False)
                .str.replace("%", "", regex=False)
                .str.replace(" ", "", regex=False)
                .str.replace(".", "", regex=False)
                .str.replace(",", ".", regex=False)
            )
            df_copy[col] = pd.to_numeric(df_copy[col], errors="coerce")
    return df_copy


def read_csv_if_exists(path, parse_dates=None):
    if os.path.exists(path):
        return pd.read_csv(path, parse_dates=parse_dates)
    return None


def export_table_files(dataframe, output_folder, file_name, sheet_name="Data"):
    os.makedirs(output_folder, exist_ok=True)
    csv_path = f"{output_folder}/{file_name}.csv"
    csv_excel_es_path = f"{output_folder}/{file_name}_excel_es.csv"
    xlsx_path = f"{output_folder}/{file_name}.xlsx"
    df_export = dataframe.copy()
    for col in df_export.columns:
        if pd.api.types.is_datetime64_any_dtype(df_export[col]):
            df_export[col] = df_export[col].dt.strftime("%d/%m/%Y %H:%M")
    dataframe.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df_export.to_csv(csv_excel_es_path, index=False, sep=";", decimal=",", encoding="utf-8-sig")
    safe_sheet_name = sheet_name[:31]
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        dataframe.to_excel(writer, index=False, sheet_name=safe_sheet_name)
        ws = writer.sheets[safe_sheet_name]
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.font = Font(bold=True)
        for col_idx, col_name in enumerate(dataframe.columns, start=1):
            letter = get_column_letter(col_idx)
            if pd.api.types.is_float_dtype(dataframe[col_name]):
                number_format = "0.00%" if any(token in col_name for token in ["pct", "rate", "score", "share"]) else "0.00"
                for cell in ws[letter][1:]:
                    cell.number_format = number_format
            elif pd.api.types.is_integer_dtype(dataframe[col_name]):
                for cell in ws[letter][1:]:
                    cell.number_format = "0"
        for column_cells in ws.columns:
            max_length = 0
            letter = get_column_letter(column_cells[0].column)
            for cell in column_cells:
                if cell.value is not None:
                    max_length = max(max_length, len(str(cell.value)))
            ws.column_dimensions[letter].width = min(max_length + 2, 45)
    print(f"Exportado: {file_name}")


def format_eur(value, decimals=0):
    if pd.isna(value):
        return "-"
    return f"{value:,.{decimals}f} €".replace(",", "X").replace(".", ",").replace("X", ".")


def format_pct(value, decimals=1):
    if pd.isna(value):
        return "-"
    return f"{value:.{decimals}%}".replace(".", ",")


def format_number(value, decimals=0):
    if pd.isna(value):
        return "-"
    return f"{value:,.{decimals}f}".replace(",", "X").replace(".", ",").replace("X", ".")

print("Funciones auxiliares creadas correctamente.")

## 3. Carga de datos del proyecto

Cargamos todos los outputs relevantes de los notebooks anteriores.

El dashboard funciona mejor si se han ejecutado los notebooks 1 a 6, pero incluye cálculos fallback para evitar bloqueos.

In [ ]:
# ============================================================
# 5. CARGA DEL DATASET PRINCIPAL
# ============================================================

df = pd.read_csv(data_path, parse_dates=["trip_date", "departure_datetime"])

numeric_cols = [
    "capacity", "base_price", "avg_ticket_price", "competitor_price",
    "price_index_vs_competitor", "route_elasticity", "days_before_departure",
    "weather_score", "event_flag", "expected_demand", "tickets_sold",
    "occupancy_rate", "revenue", "fixed_operational_cost",
    "variable_cost_per_passenger", "total_operational_cost", "margin", "margin_pct"
]

df = force_numeric_columns(df, numeric_cols)

print("Dataset principal cargado.")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")

df.head()

In [ ]:
# ============================================================
# 6. CARGA DE OUTPUTS DE NOTEBOOKS ANTERIORES
# ============================================================

paths = {
    "elasticity_comparison": f"{reports_path}/elasticity_comparison_by_route.csv",
    "model_leaderboard": f"{reports_path}/demand_model_leaderboard.csv",
    "model_route_error": f"{reports_path}/demand_model_route_error_summary.csv",
    "scenario_global": f"{reports_path}/pricing_simulation_global_summary.csv",
    "scenario_route": f"{reports_path}/pricing_simulation_route_scenario_summary.csv",
    "recommender_trip": f"{reports_path}/pricing_recommender_trip_recommendations.csv",
    "recommender_route": f"{reports_path}/pricing_recommender_route_summary.csv",
    "recommender_action": f"{reports_path}/pricing_recommender_action_distribution.csv",
    "recommender_manual": f"{reports_path}/pricing_recommender_manual_review.csv",
    "recommender_high_priority": f"{reports_path}/pricing_recommender_high_priority.csv"
}

elasticity_comparison = read_csv_if_exists(paths["elasticity_comparison"])
model_leaderboard = read_csv_if_exists(paths["model_leaderboard"])
model_route_error = read_csv_if_exists(paths["model_route_error"])
scenario_global = read_csv_if_exists(paths["scenario_global"])
scenario_route = read_csv_if_exists(paths["scenario_route"])
recommender_trip = read_csv_if_exists(paths["recommender_trip"], parse_dates=["trip_date", "departure_datetime"])
recommender_route = read_csv_if_exists(paths["recommender_route"])
recommender_action = read_csv_if_exists(paths["recommender_action"])
recommender_manual = read_csv_if_exists(paths["recommender_manual"])
recommender_high_priority = read_csv_if_exists(paths["recommender_high_priority"])

loaded_status = pd.DataFrame({
    "dataset": list(paths.keys()),
    "path": list(paths.values()),
    "loaded": [
        elasticity_comparison is not None,
        model_leaderboard is not None,
        model_route_error is not None,
        scenario_global is not None,
        scenario_route is not None,
        recommender_trip is not None,
        recommender_route is not None,
        recommender_action is not None,
        recommender_manual is not None,
        recommender_high_priority is not None
    ]
})

loaded_status

## 4. Tablas fallback

Si no existiera alguna tabla del recomendador, generamos una versión básica para que el dashboard pueda ejecutarse.

In [ ]:
# ============================================================
# 7. PERFORMANCE BASE POR RUTA
# ============================================================

base_route_performance = (
    df
    .groupby("route")
    .agg(
        trips=("trip_id", "count"),
        total_tickets=("tickets_sold", "sum"),
        total_revenue=("revenue", "sum"),
        total_margin=("margin", "sum"),
        avg_occupancy=("occupancy_rate", "mean"),
        avg_ticket_price=("avg_ticket_price", "mean"),
        avg_margin_pct=("margin_pct", "mean"),
        avg_price_index=("price_index_vs_competitor", "mean"),
        avg_elasticity=("route_elasticity", "mean")
    )
    .reset_index()
)

base_route_performance["margin_over_revenue"] = np.where(
    base_route_performance["total_revenue"] > 0,
    base_route_performance["total_margin"] / base_route_performance["total_revenue"],
    0
)

base_route_performance = base_route_performance.sort_values("total_revenue", ascending=False)

base_route_performance

In [ ]:
# ============================================================
# 8. FALLBACK DEL RECOMENDADOR
# ============================================================

if recommender_route is None:
    print("No se encontró resumen del recomendador. Se crea fallback desde performance base.")
    recommender_route = base_route_performance.copy()
    recommender_route["avg_current_price"] = recommender_route["avg_ticket_price"]
    recommender_route["avg_recommended_price"] = recommender_route["avg_ticket_price"]
    recommender_route["avg_price_change_pct"] = 0
    recommender_route["avg_current_occupancy"] = recommender_route["avg_occupancy"]
    recommender_route["avg_recommended_occupancy"] = recommender_route["avg_occupancy"]
    recommender_route["current_revenue"] = recommender_route["total_revenue"]
    recommender_route["recommended_revenue"] = recommender_route["total_revenue"]
    recommender_route["revenue_uplift"] = 0
    recommender_route["revenue_uplift_pct"] = 0
    recommender_route["current_margin"] = recommender_route["total_margin"]
    recommender_route["recommended_margin"] = recommender_route["total_margin"]
    recommender_route["margin_uplift"] = 0
    recommender_route["margin_uplift_pct"] = 0
    recommender_route["avg_current_margin_pct"] = recommender_route["avg_margin_pct"]
    recommender_route["avg_recommended_margin_pct"] = recommender_route["avg_margin_pct"]
    recommender_route["avg_elasticity"] = recommender_route["avg_elasticity"]
    recommender_route["confidence_score"] = 0.50
    recommender_route["confidence_level"] = "Fallback"
    recommender_route["dominant_pricing_action"] = "Maintain / monitor"
    recommender_route["dominant_action_share"] = 1.0

if recommender_action is None:
    recommender_action = pd.DataFrame({
        "pricing_action": ["Maintain / monitor"],
        "number_of_trips": [len(df)],
        "share_of_trips": [1.0]
    })

if recommender_trip is None:
    recommender_trip = df.copy()
    recommender_trip["recommended_ticket_price"] = recommender_trip["avg_ticket_price"]
    recommender_trip["recommended_price_change_pct"] = 0
    recommender_trip["recommended_occupancy_rate"] = recommender_trip["occupancy_rate"]
    recommender_trip["recommended_revenue"] = recommender_trip["revenue"]
    recommender_trip["recommended_margin"] = recommender_trip["margin"]
    recommender_trip["revenue_uplift"] = 0
    recommender_trip["revenue_uplift_pct"] = 0
    recommender_trip["margin_uplift"] = 0
    recommender_trip["margin_uplift_pct"] = 0
    recommender_trip["recommendation_score"] = 0.50
    recommender_trip["confidence_level"] = "Fallback"
    recommender_trip["pricing_action"] = "Maintain / monitor"
    recommender_trip["recommendation_explanation"] = "Fallback recommendation based on current price."

print("Tablas fallback preparadas correctamente.")

## 5. KPIs ejecutivos

Calculamos la fotografía global del negocio y el impacto estimado de las recomendaciones.

In [ ]:
# ============================================================
# 9. KPIS EJECUTIVOS
# ============================================================

total_trips = df["trip_id"].nunique()
total_tickets = df["tickets_sold"].sum()
total_revenue = df["revenue"].sum()
total_margin = df["margin"].sum()
avg_occupancy = df["occupancy_rate"].mean()
avg_ticket_price = df["avg_ticket_price"].mean()
avg_margin_pct = df["margin_pct"].mean()
avg_price_index = df["price_index_vs_competitor"].mean()

recommended_revenue = recommender_trip["recommended_revenue"].sum()
recommended_margin = recommender_trip["recommended_margin"].sum()
estimated_revenue_uplift = recommender_trip["revenue_uplift"].sum()
estimated_margin_uplift = recommender_trip["margin_uplift"].sum()

estimated_revenue_uplift_pct = estimated_revenue_uplift / total_revenue if total_revenue != 0 else 0
estimated_margin_uplift_pct = estimated_margin_uplift / total_margin if total_margin != 0 else 0

high_confidence_recs = (recommender_trip["confidence_level"] == "High").sum()
manual_review_recs = recommender_trip[
    recommender_trip["pricing_action"].astype(str).str.contains("review", case=False, na=False) |
    recommender_trip["confidence_level"].astype(str).str.contains("Manual", case=False, na=False)
].shape[0]

executive_kpis = pd.DataFrame({
    "kpi": [
        "Trips", "Tickets sold", "Current revenue", "Current margin", "Average occupancy",
        "Average ticket price", "Average margin %", "Average price index vs competitor",
        "Recommended revenue", "Recommended margin", "Estimated revenue uplift",
        "Estimated margin uplift", "High confidence recommendations", "Manual review recommendations"
    ],
    "value": [
        total_trips, total_tickets, total_revenue, total_margin, avg_occupancy,
        avg_ticket_price, avg_margin_pct, avg_price_index, recommended_revenue,
        recommended_margin, estimated_revenue_uplift, estimated_margin_uplift,
        high_confidence_recs, manual_review_recs
    ]
})

executive_kpis

## 6. Tablas ejecutivas para dashboard

Preparamos las tablas que alimentan el HTML final.

In [ ]:
# ============================================================
# 10. TABLA EJECUTIVA POR RUTA
# ============================================================

route_dashboard_table = recommender_route.copy()

route_dashboard_table = force_numeric_columns(
    route_dashboard_table,
    [
        "trips", "avg_current_price", "avg_recommended_price", "avg_price_change_pct",
        "avg_current_occupancy", "avg_recommended_occupancy", "current_revenue",
        "recommended_revenue", "revenue_uplift", "revenue_uplift_pct", "current_margin",
        "recommended_margin", "margin_uplift", "margin_uplift_pct", "avg_current_margin_pct",
        "avg_recommended_margin_pct", "avg_elasticity", "confidence_score", "dominant_action_share"
    ]
)

route_dashboard_table = route_dashboard_table.sort_values("revenue_uplift_pct", ascending=False)

route_dashboard_table.head()

In [ ]:
# ============================================================
# 11. TOP RECOMENDACIONES Y CASOS DE REVISIÓN
# ============================================================

top_recommendations = (
    recommender_trip
    .sort_values("recommendation_score", ascending=False)
    .head(25)
    .copy()
)

manual_review_table = recommender_trip[
    recommender_trip["pricing_action"].astype(str).str.contains("review", case=False, na=False) |
    recommender_trip["confidence_level"].astype(str).str.contains("Manual|Low", case=False, na=False)
].copy()

manual_review_table = manual_review_table.sort_values(
    ["recommendation_score", "margin_uplift_pct"],
    ascending=[True, True]
).head(25)

print("Top recomendaciones:", top_recommendations.shape)
print("Casos revisión:", manual_review_table.shape)

## 7. Visualizaciones finales

Creamos visualizaciones ejecutivas para el dashboard y para documentación del README.

In [ ]:
# ============================================================
# 12. REVENUE Y MARGEN POR RUTA
# ============================================================

route_financial_plot = base_route_performance[["route", "total_revenue", "total_margin"]].copy()
route_financial_plot = route_financial_plot.melt(id_vars="route", value_vars=["total_revenue", "total_margin"], var_name="metric", value_name="value")
plt.figure(figsize=(13, 7))
sns.barplot(data=route_financial_plot, x="value", y="route", hue="metric")
plt.title("Revenue y margen actual por ruta")
plt.xlabel("€")
plt.ylabel("Ruta")
plt.tight_layout()
plt.savefig(f"{images_path}/45_dashboard_revenue_margin_by_route.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 13. CAMBIO DE PRECIO RECOMENDADO POR RUTA
# ============================================================

plot_df = route_dashboard_table.sort_values("avg_price_change_pct", ascending=False).copy()
plt.figure(figsize=(13, 7))
sns.barplot(data=plot_df, x="avg_price_change_pct", y="route")
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Cambio medio de precio recomendado por ruta")
plt.xlabel("Cambio medio recomendado")
plt.ylabel("Ruta")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()
plt.savefig(f"{images_path}/46_dashboard_price_change_by_route.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 14. ACCIONES RECOMENDADAS
# ============================================================

plt.figure(figsize=(12, 6))
sns.barplot(data=recommender_action.sort_values("number_of_trips", ascending=False), x="number_of_trips", y="pricing_action")
plt.title("Distribución de acciones recomendadas")
plt.xlabel("Número de viajes")
plt.ylabel("Acción")
plt.tight_layout()
plt.savefig(f"{images_path}/47_dashboard_pricing_actions.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 15. HEATMAP DE ACCIONES POR RUTA
# ============================================================

action_route_pivot = pd.crosstab(recommender_trip["route"], recommender_trip["pricing_action"], normalize="index")
plt.figure(figsize=(13, 7))
sns.heatmap(action_route_pivot, annot=True, fmt=".1%", cmap="Blues")
plt.title("Distribución de acciones recomendadas por ruta")
plt.xlabel("Acción recomendada")
plt.ylabel("Ruta")
plt.tight_layout()
plt.savefig(f"{images_path}/48_dashboard_actions_by_route_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 16. ELASTICIDAD VS OCUPACIÓN
# ============================================================

elasticity_plot = base_route_performance.copy()
if elasticity_comparison is not None and "elasticity_for_simulation" in elasticity_comparison.columns:
    elasticity_plot = elasticity_plot.merge(elasticity_comparison[["route", "elasticity_for_simulation"]], on="route", how="left")
else:
    elasticity_plot["elasticity_for_simulation"] = elasticity_plot["avg_elasticity"]
plt.figure(figsize=(10, 7))
sns.scatterplot(data=elasticity_plot, x="elasticity_for_simulation", y="avg_occupancy", size="total_revenue", sizes=(100, 700), hue="route")
plt.axhline(0.85, linestyle="--", linewidth=1)
plt.axhline(0.55, linestyle="--", linewidth=1)
plt.axvline(-1, linestyle="--", linewidth=1)
plt.title("Mapa ejecutivo: elasticidad vs ocupación por ruta")
plt.xlabel("Elasticidad precio-demanda")
plt.ylabel("Ocupación media")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()
plt.savefig(f"{images_path}/49_dashboard_elasticity_vs_occupancy.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Preparación de datos para HTML

Transformamos las tablas principales a JSON para generar un dashboard HTML autónomo.

In [ ]:
# ============================================================
# 17. PREPARAR DATOS PARA DASHBOARD HTML
# ============================================================

route_html_columns = [
    "route", "trips", "avg_current_price", "avg_recommended_price", "avg_price_change_pct",
    "avg_current_occupancy", "avg_recommended_occupancy", "current_revenue", "recommended_revenue",
    "revenue_uplift_pct", "current_margin", "recommended_margin", "margin_uplift_pct",
    "avg_elasticity", "confidence_score", "confidence_level", "dominant_pricing_action", "dominant_action_share"
]
route_html_columns = [col for col in route_html_columns if col in route_dashboard_table.columns]
route_html = route_dashboard_table[route_html_columns].copy()

action_html = recommender_action.copy()

top_html_columns = [
    "trip_id", "trip_date", "route", "season", "day_of_week", "avg_ticket_price",
    "recommended_ticket_price", "recommended_price_change_pct", "recommended_occupancy_rate",
    "revenue_uplift_pct", "margin_uplift_pct", "recommendation_score", "confidence_level",
    "pricing_action", "recommendation_explanation"
]
top_html_columns = [col for col in top_html_columns if col in top_recommendations.columns]
top_html = top_recommendations[top_html_columns].copy()

if "trip_date" in top_html.columns:
    top_html["trip_date"] = top_html["trip_date"].astype(str)

route_json = json.dumps(route_html.to_dict(orient="records"), ensure_ascii=False)
action_json = json.dumps(action_html.to_dict(orient="records"), ensure_ascii=False)
top_json = json.dumps(top_html.to_dict(orient="records"), ensure_ascii=False)

print("JSON preparado para dashboard HTML.")

## 9. Generación del dashboard ejecutivo HTML

Creamos el dashboard final:

`dashboard/executive_pricing_dashboard.html`

Este archivo se podrá abrir directamente en cualquier navegador.

In [ ]:
# ============================================================
# 18. GENERAR DASHBOARD EJECUTIVO HTML
# ============================================================

kpi_cards_html = f"""
<div class='kpi-card'><div class='kpi-title'>Revenue actual</div><div class='kpi-value'>{format_eur(total_revenue)}</div><div class='kpi-subtitle'>Ingresos totales del dataset</div></div>
<div class='kpi-card'><div class='kpi-title'>Margen actual</div><div class='kpi-value'>{format_eur(total_margin)}</div><div class='kpi-subtitle'>Margen medio: {format_pct(avg_margin_pct)}</div></div>
<div class='kpi-card'><div class='kpi-title'>Ocupación media</div><div class='kpi-value'>{format_pct(avg_occupancy)}</div><div class='kpi-subtitle'>Tickets: {format_number(total_tickets)}</div></div>
<div class='kpi-card'><div class='kpi-title'>Uplift revenue estimado</div><div class='kpi-value'>{format_eur(estimated_revenue_uplift)}</div><div class='kpi-subtitle'>{format_pct(estimated_revenue_uplift_pct)}</div></div>
<div class='kpi-card'><div class='kpi-title'>Uplift margen estimado</div><div class='kpi-value'>{format_eur(estimated_margin_uplift)}</div><div class='kpi-subtitle'>{format_pct(estimated_margin_uplift_pct)}</div></div>
<div class='kpi-card'><div class='kpi-title'>Precio medio</div><div class='kpi-value'>{format_eur(avg_ticket_price, 2)}</div><div class='kpi-subtitle'>Índice competidor: {avg_price_index:.2f}</div></div>
<div class='kpi-card'><div class='kpi-title'>Alta confianza</div><div class='kpi-value'>{format_number(high_confidence_recs)}</div><div class='kpi-subtitle'>Recomendaciones High</div></div>
<div class='kpi-card'><div class='kpi-title'>Revisión manual</div><div class='kpi-value'>{format_number(manual_review_recs)}</div><div class='kpi-subtitle'>Casos con riesgo</div></div>
"""

html_template = """
<!doctype html>
<html lang='es'>
<head>
<meta charset='utf-8'>
<meta name='viewport' content='width=device-width, initial-scale=1'>
<title>Levante Ferries — Executive Pricing Dashboard</title>
<style>
body{margin:0;background:#f3f7fb;color:#102a43;font-family:Arial,sans-serif}
main{max-width:1280px;margin:0 auto;padding:36px 22px}.eyebrow{color:#0b4f6c;font-weight:800;text-transform:uppercase;letter-spacing:.13em;font-size:12px}h1{font-size:42px;margin:8px 0;letter-spacing:-.045em}.subtitle{max-width:820px;color:#627d98;line-height:1.55}.kpi-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:22px 0}.kpi-card,.section{background:white;border:1px solid #d9e2ec;border-radius:18px;padding:18px;box-shadow:0 18px 45px rgba(16,42,67,.08)}.kpi-title{color:#627d98;font-size:13px;font-weight:700}.kpi-value{font-size:28px;font-weight:850;margin:7px 0 4px}.kpi-subtitle{color:#627d98;font-size:12px}.section{margin:18px 0}.grid-2{display:grid;grid-template-columns:1fr 1fr;gap:18px}.controls{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-bottom:16px}label{color:#627d98;font-size:13px;font-weight:700}select{width:100%;margin-top:6px;padding:10px;border:1px solid #d9e2ec;border-radius:10px;background:white}.chart{width:100%;min-height:320px}table{width:100%;border-collapse:collapse;font-size:13px}th,td{border-bottom:1px solid #e5eaf0;padding:8px;text-align:left;vertical-align:top}th{background:#f3f7fb}.badge{display:inline-block;padding:4px 8px;border-radius:999px;font-size:12px;font-weight:800;background:#fef3c7;color:#92400e}.badge-increase-price{background:#d1fae5;color:#065f46}.badge-promotional-action{background:#dbeafe;color:#1e40af}.badge-manual-review,.badge-margin-review,.badge-capacity---premium-review{background:#fee2e2;color:#991b1b}.tool-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:14px}.tool-card{border:1px solid #d9e2ec;border-radius:16px;padding:16px;background:#fbfdff}.tool-card p{color:#627d98}.footer{color:#627d98;text-align:center;margin-top:24px;font-size:12px}@media(max-width:950px){.kpi-grid,.grid-2,.controls,.tool-grid{grid-template-columns:1fr}}
</style>
</head>
<body>
<main>
<div class='eyebrow'>Levante Ferries · Pricing & Revenue Management</div>
<h1>Executive Pricing Dashboard</h1>
<p class='subtitle'>Dashboard final del proyecto de optimización de pricing. Integra demanda, elasticidad, margen, simulación de escenarios y recomendaciones explicables por ruta y viaje.</p>
<section class='kpi-grid'>__KPI_CARDS__</section>
<section class='section'><h2>Filtros ejecutivos</h2><div class='controls'><label>Ruta<select id='routeFilter'></select></label><label>Acción<select id='actionFilter'></select></label><label>Confianza<select id='confidenceFilter'></select></label></div></section>
<section class='grid-2'><div class='section'><h2>Revenue uplift por ruta</h2><svg id='routeRevenueChart' class='chart' viewBox='0 0 620 360'></svg></div><div class='section'><h2>Acciones recomendadas</h2><svg id='actionChart' class='chart' viewBox='0 0 620 360'></svg></div></section>
<section class='grid-2'><div class='section'><h2>Cambio de precio recomendado</h2><svg id='priceChangeChart' class='chart' viewBox='0 0 620 360'></svg></div><div class='section'><h2>Margen uplift por ruta</h2><svg id='marginChart' class='chart' viewBox='0 0 620 360'></svg></div></section>
<section class='section'><h2>Resumen ejecutivo por ruta</h2><table><thead><tr><th>Ruta</th><th>Trips</th><th>Precio actual</th><th>Precio recomendado</th><th>Cambio</th><th>Revenue uplift</th><th>Margin uplift</th><th>Acción dominante</th><th>Confianza</th></tr></thead><tbody id='routeTableBody'></tbody></table></section>
<section class='section'><h2>Top recomendaciones por viaje</h2><table><thead><tr><th>Viaje</th><th>Fecha</th><th>Ruta</th><th>Temporada</th><th>Precio actual</th><th>Precio recomendado</th><th>Cambio</th><th>Score</th><th>Acción</th><th>Explicación</th></tr></thead><tbody id='topTableBody'></tbody></table></section>
<section class='section'><h2>Herramientas interactivas del proyecto</h2><div class='tool-grid'><div class='tool-card'><h3>Pricing Simulator</h3><p>Simula cambios de precio, demanda externa y shocks de coste.</p><code>pricing_interactive_simulator.html</code></div><div class='tool-card'><h3>Break-even Tool</h3><p>Calcula punto de equilibrio, precio mínimo y ocupación mínima rentable.</p><code>break_even_pricing_decision_tool.html</code></div><div class='tool-card'><h3>Recommendation Explorer</h3><p>Explora recomendaciones por viaje, ruta, temporada, acción y confianza.</p><code>pricing_recommendation_explorer.html</code></div></div></section>
<div class='footer'>Proyecto simulado de portfolio · Levante Ferries · Datos sintéticos · Business Analytics & Pricing</div>
</main>
<script>
const routeData = __ROUTE_DATA__;
const actionData = __ACTION_DATA__;
const topData = __TOP_DATA__;
const euro = v => new Intl.NumberFormat('es-ES',{style:'currency',currency:'EUR',maximumFractionDigits:0}).format(v || 0);
const euro2 = v => new Intl.NumberFormat('es-ES',{style:'currency',currency:'EUR',maximumFractionDigits:2}).format(v || 0);
const pct = v => new Intl.NumberFormat('es-ES',{style:'percent',maximumFractionDigits:1,signDisplay:'exceptZero'}).format(v || 0);
const num = v => new Intl.NumberFormat('es-ES',{maximumFractionDigits:0}).format(v || 0);
function uniqueValues(data, field){return [...new Set(data.map(d => d[field]).filter(Boolean))].sort();}
function fillSelect(id, values){const s=document.getElementById(id);s.innerHTML='';s.add(new Option('All','All'));values.forEach(v=>s.add(new Option(v,v)));}
function badgeClass(action){return 'badge-' + String(action).toLowerCase().replaceAll(' ','-').replaceAll('/','-');}
function barChart(svgId, data, labelField, valueField, formatter, color){const svg=document.getElementById(svgId);const width=620, ml=170, iw=390, rh=34;const sorted=data.slice().sort((a,b)=>(b[valueField]||0)-(a[valueField]||0)).slice(0,8);const maxAbs=Math.max(...sorted.map(d=>Math.abs(d[valueField]||0)),0.001);let html='';sorted.forEach((d,i)=>{const y=22+i*rh;const value=d[valueField]||0;const bw=Math.abs(value)/maxAbs*iw;html+=`<text x='8' y='${y+18}' fill='#102a43' font-size='12' font-weight='700'>${d[labelField]}</text>`;html+=`<rect x='${ml}' y='${y}' width='${bw}' height='20' rx='5' fill='${color}' opacity='${value<0?0.45:1}'></rect>`;html+=`<text x='${Math.min(ml+bw+8,width-80)}' y='${y+15}' fill='#102a43' font-size='12'>${formatter(value)}</text>`;});svg.innerHTML=html;}
function actionBarChart(svgId, data){const svg=document.getElementById(svgId);const ml=190, iw=365, rh=40;const sorted=data.slice().sort((a,b)=>b.number_of_trips-a.number_of_trips);const max=Math.max(...sorted.map(d=>d.number_of_trips),1);let html='';sorted.forEach((d,i)=>{const y=24+i*rh;const w=d.number_of_trips/max*iw;html+=`<text x='8' y='${y+17}' fill='#102a43' font-size='12' font-weight='700'>${d.pricing_action}</text>`;html+=`<rect x='${ml}' y='${y}' width='${w}' height='22' rx='5' fill='#168aad'></rect>`;html+=`<text x='${Math.min(ml+w+8,550)}' y='${y+16}' fill='#102a43' font-size='12'>${num(d.number_of_trips)}</text>`;});svg.innerHTML=html;}
function renderTables(routes,trips){document.getElementById('routeTableBody').innerHTML=routes.sort((a,b)=>(b.revenue_uplift_pct||0)-(a.revenue_uplift_pct||0)).map(d=>`<tr><td><b>${d.route}</b></td><td>${num(d.trips)}</td><td>${euro2(d.avg_current_price)}</td><td>${euro2(d.avg_recommended_price)}</td><td>${pct(d.avg_price_change_pct)}</td><td>${pct(d.revenue_uplift_pct)}</td><td>${pct(d.margin_uplift_pct)}</td><td><span class='badge ${badgeClass(d.dominant_pricing_action)}'>${d.dominant_pricing_action}</span></td><td>${d.confidence_level}</td></tr>`).join('');document.getElementById('topTableBody').innerHTML=trips.sort((a,b)=>(b.recommendation_score||0)-(a.recommendation_score||0)).slice(0,30).map(d=>`<tr><td>${d.trip_id}</td><td>${d.trip_date}</td><td>${d.route}</td><td>${d.season}</td><td>${euro2(d.avg_ticket_price)}</td><td>${euro2(d.recommended_ticket_price)}</td><td>${pct(d.recommended_price_change_pct)}</td><td>${pct(d.recommendation_score)}</td><td><span class='badge ${badgeClass(d.pricing_action)}'>${d.pricing_action}</span></td><td>${d.recommendation_explanation||''}</td></tr>`).join('');}
function applyFilters(){const route=document.getElementById('routeFilter').value;const action=document.getElementById('actionFilter').value;const confidence=document.getElementById('confidenceFilter').value;let filteredRoutes=routeData.slice();let filteredTop=topData.slice();if(route!=='All'){filteredRoutes=filteredRoutes.filter(d=>d.route===route);filteredTop=filteredTop.filter(d=>d.route===route);}if(action!=='All'){filteredRoutes=filteredRoutes.filter(d=>d.dominant_pricing_action===action);filteredTop=filteredTop.filter(d=>d.pricing_action===action);}if(confidence!=='All'){filteredRoutes=filteredRoutes.filter(d=>d.confidence_level===confidence);filteredTop=filteredTop.filter(d=>d.confidence_level===confidence);}renderTables(filteredRoutes,filteredTop);barChart('routeRevenueChart',filteredRoutes,'route','revenue_uplift_pct',pct,'#168aad');barChart('priceChangeChart',filteredRoutes,'route','avg_price_change_pct',pct,'#0b4f6c');barChart('marginChart',filteredRoutes,'route','margin_uplift_pct',pct,'#147d64');}
fillSelect('routeFilter',uniqueValues(routeData,'route'));fillSelect('actionFilter',uniqueValues(routeData,'dominant_pricing_action'));fillSelect('confidenceFilter',uniqueValues(routeData,'confidence_level'));['routeFilter','actionFilter','confidenceFilter'].forEach(id=>document.getElementById(id).addEventListener('change',applyFilters));actionBarChart('actionChart',actionData);applyFilters();
</script>
</body>
</html>
"""

dashboard_html = (
    html_template
    .replace("__KPI_CARDS__", kpi_cards_html)
    .replace("__ROUTE_DATA__", route_json)
    .replace("__ACTION_DATA__", action_json)
    .replace("__TOP_DATA__", top_json)
)

dashboard_output_path = f"{dashboard_path}/executive_pricing_dashboard.html"

with open(dashboard_output_path, "w", encoding="utf-8") as file:
    file.write(dashboard_html)

print("Dashboard ejecutivo HTML creado en:")
print(dashboard_output_path)

## 10. Exportación de tablas finales

Exportamos los outputs finales del dashboard.

In [ ]:
# ============================================================
# 19. EXPORTACIÓN DE TABLAS FINALES
# ============================================================

tables_to_export = {
    "dashboard_executive_kpis": executive_kpis,
    "dashboard_route_table": route_dashboard_table,
    "dashboard_top_recommendations": top_recommendations,
    "dashboard_manual_review_table": manual_review_table,
    "dashboard_loaded_status": loaded_status
}

for file_name, table in tables_to_export.items():
    export_table_files(table, reports_path, file_name, sheet_name=file_name[:31])

print("Tablas finales del dashboard exportadas correctamente.")

## 11. Resumen ejecutivo final del proyecto

Generamos un resumen final para documentación y README.

In [ ]:
# ============================================================
# 20. RESUMEN EJECUTIVO FINAL EN MARKDOWN
# ============================================================

top_revenue_route = route_dashboard_table.sort_values("revenue_uplift_pct", ascending=False).iloc[0]
top_margin_route = route_dashboard_table.sort_values("margin_uplift_pct", ascending=False).iloc[0]
most_common_action = recommender_action.sort_values("number_of_trips", ascending=False).iloc[0]

final_summary = f"""
# Final Executive Summary — Dynamic Pricing & Revenue Optimization

## Project

Levante Ferries — Dynamic Pricing & Revenue Optimization for Ferry Routes.

## Objective

Build a complete decision-support system for ferry pricing, combining demand, elasticity, margin, occupancy, scenario simulation and explainable pricing recommendations.

## Current performance

- Trips analyzed: {total_trips:,.0f}
- Tickets sold: {total_tickets:,.0f}
- Current revenue: {total_revenue:,.2f} €
- Current margin: {total_margin:,.2f} €
- Average occupancy: {avg_occupancy:.2%}
- Average ticket price: {avg_ticket_price:.2f} €

## Recommendation impact

- Estimated recommended revenue: {recommended_revenue:,.2f} €
- Estimated recommended margin: {recommended_margin:,.2f} €
- Estimated revenue uplift: {estimated_revenue_uplift:,.2f} € ({estimated_revenue_uplift_pct:.2%})
- Estimated margin uplift: {estimated_margin_uplift:,.2f} € ({estimated_margin_uplift_pct:.2%})

## Key insights

- Main recommended action: {most_common_action['pricing_action']}
- Top revenue opportunity route: {top_revenue_route['route']} with {top_revenue_route['revenue_uplift_pct']:.2%} estimated uplift
- Top margin opportunity route: {top_margin_route['route']} with {top_margin_route['margin_uplift_pct']:.2%} estimated uplift
- High-confidence recommendations: {high_confidence_recs:,.0f}
- Manual review recommendations: {manual_review_recs:,.0f}

## Methodology

1. Synthetic business dataset generation.
2. Exploratory analysis of pricing, demand and margin.
3. Price-demand elasticity estimation.
4. Demand prediction model.
5. Advanced pricing scenario simulation.
6. Formal pricing recommendation engine.
7. Executive dashboard.

## Business conclusion

Dynamic pricing should not be treated as a simple price increase problem.  
The right pricing decision depends on occupancy, elasticity, margin, cost structure and competitive positioning.
"""

final_summary_path = f"{reports_path}/final_executive_summary_project_03_pricing.md"

with open(final_summary_path, "w", encoding="utf-8") as file:
    file.write(final_summary)

print("Resumen ejecutivo final guardado en:")
print(final_summary_path)

## 12. Archivos finales generados

Listamos los archivos clave del proyecto.

In [ ]:
# ============================================================
# 21. LISTADO DE ARCHIVOS FINALES
# ============================================================

print("Dashboard final:")
for file in sorted(os.listdir(dashboard_path)):
    if file.endswith(".html"):
        print("-", file)

print("
Imágenes finales del dashboard:")
for file in sorted(os.listdir(images_path)):
    if file.endswith(".png") and file.startswith(("45_", "46_", "47_", "48_", "49_")):
        print("-", file)

print("
Reportes finales:")
for file in sorted(os.listdir(reports_path)):
    if file.startswith("dashboard_") or file == "final_executive_summary_project_03_pricing.md":
        print("-", file)

## Conclusiones del Notebook 7

En este notebook hemos construido la capa final del proyecto:

- KPIs ejecutivos.
- Tablas finales.
- Gráficos ejecutivos.
- Resumen por ruta.
- Top recomendaciones.
- Herramientas interactivas enlazadas.
- Dashboard HTML autónomo.
- Resumen ejecutivo final.

Este notebook completa el proyecto de Optimización de Pricing.

La narrativa final es:

> De datos sintéticos realistas a decisiones de pricing explicables, apoyadas en demanda, elasticidad, ocupación, margen, competencia y simulación de escenarios.

Con esto el proyecto queda preparado para GitHub, README, portfolio y presentación profesional.